In [ ]:
import os
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
os.environ.setdefault("CHROMA_TELEMETRY_DISABLED", "true")

import chromadb
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader

c:\Users\Priyanshi Garg\AI_Training_Batch_May_2026\submissions\assignments\assignment-03\priyanshi-garg\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from dotenv import load_dotenv
load_dotenv()
import os
os.environ['NVIDIA_API_KEY'] = os.getenv('NVIDIA_API_KEY')

In [4]:
from openai import OpenAI
client = OpenAI(
    api_key=os.environ['NVIDIA_API_KEY'],
    base_url="https://integrate.api.nvidia.com/v1"
)

In [5]:
model = "meta/llama-3.1-70b-instruct"

In [14]:
pdf_folder = os.path.abspath(r"C:\Users\Priyanshi Garg\Downloads\tesla-annual-reports\tesla-annual-reports")
pdf_files = sorted([
    os.path.join(pdf_folder, file_name)
    for file_name in os.listdir(pdf_folder)
    if file_name.lower().endswith(".pdf")
])

print(f"Found {len(pdf_files)} PDF files in {pdf_folder}")

Found 5 PDF files in C:\Users\Priyanshi Garg\Downloads\tesla-annual-reports\tesla-annual-reports


In [15]:
pdf_loader = PyPDFDirectoryLoader(pdf_folder)

In [16]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap=16
)

In [17]:
from langchain_community.document_loaders import PyPDFLoader

In [19]:
import re

tesla_chunks = []

for file in pdf_files:
    match = re.search(r"(\d{4})", os.path.basename(file))
    if not match:
        raise ValueError(f"Could not extract a 4-digit year from filename: {os.path.basename(file)}")

    year = int(match.group(1))

    pages = PyPDFLoader(file).load()
    chunks = text_splitter.split_documents(pages)

    for idx, chunk in enumerate(chunks):
        chunk.metadata.update({
            "chunk_id": f"tsla_{year}_{idx}",
            "source_doc": file,
            "company": "Tesla, Inc.",
            "fiscal_year": year,
            "form_type": "10-K"
        })
        tesla_chunks.append(chunk)


In [20]:
tesla_chunks[0]

Document(metadata={'producer': 'Qt 5.11.3', 'creator': 'wkhtmltopdf 0.12.5', 'creationdate': '2020-09-30T05:19:35+00:00', 'title': '', 'source': 'C:\\Users\\Priyanshi Garg\\Downloads\\tesla-annual-reports\\tesla-annual-reports\\tsla-10k_20191231-gen_0.pdf', 'total_pages': 297, 'page': 0, 'page_label': '1', 'chunk_id': 'tsla_2019_0', 'source_doc': 'C:\\Users\\Priyanshi Garg\\Downloads\\tesla-annual-reports\\tesla-annual-reports\\tsla-10k_20191231-gen_0.pdf', 'company': 'Tesla, Inc.', 'fiscal_year': 2019, 'form_type': '10-K'}, page_content='UNITED\tSTATES\nSECURITIES\tAND\tEXCHANGE\tCOMMISSION\nWashington,\tD.C.\t20549\n\t\nFORM\t\n10-K\n\t\n(Mark\tOne)\n☒\nANNUAL\tREPORT\tPURSUANT\tTO\tSECTION\t13\tOR\t15(d)\tOF\tTHE\tSECURITIES\tEXCHANGE\tACT\tOF\t1934\nFor\tthe\tfiscal\tyear\tended\t\nDecember\t31,\t2019\n\t\nOR\n☐\nTRANSITION\tREPORT\tPURSUANT\tTO\tSECTION\t13\tOR\t15(d)\tOF\tTHE\tSECURITIES\tEXCHANGE\tACT\tOF\t1934\nFor\tthe\ttransition\tperiod\tfrom\t\n\t\t\t\t\t\t\t\t\t\t\t\t\t\t\

In [33]:
len(tesla_chunks)

3337

In [ ]:
# tesla_10k_collection = 'tesla-10k-2019-to-2023'

In [22]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

C:\Users\Priyanshi Garg\AppData\Local\Temp\ipykernel_7144\3270939176.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")


In [ ]:
# chromadb_client = chromadb.PersistentClient(
#     path="./tesla_db"
# )

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [26]:
import time

CHUNK_BATCH_SIZE = 500

chunk_vectorstore = Chroma(
    collection_name="tesla_chunks",
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding_model,
    client=chromadb.PersistentClient(path="./chroma_chunk_db"),
    persist_directory="./chroma_chunk_db"
)

for start in range(0, len(tesla_chunks), CHUNK_BATCH_SIZE):
    batch = tesla_chunks[start:start + CHUNK_BATCH_SIZE]
    ids = [f"tesla_chunk_{start + idx}" for idx in range(len(batch))]
    chunk_vectorstore.add_documents(documents=batch, ids=ids)
    time.sleep(0.5)

print(f"Indexed {len(tesla_chunks)} chunks in batches of {CHUNK_BATCH_SIZE}.")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Indexed 3337 chunks in batches of 500.


### Baseline Retrieval Test

In [27]:
query = """
What are the major risks that could affect Tesla's production and delivery targets?
"""

results = chunk_vectorstore.similarity_search(
    query,
    k=5
)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [29]:
results[0]

Document(id='tesla_chunk_759', metadata={'chunk_id': 'tsla_2020_41', 'company': 'Tesla, Inc.', 'creationdate': '2021-02-08T12:40:35+00:00', 'creator': 'wkhtmltopdf 0.12.5', 'fiscal_year': 2020, 'form_type': '10-K', 'page': 13, 'page_label': '14', 'producer': 'Qt 5.11.3', 'source': 'C:\\Users\\Priyanshi Garg\\Downloads\\tesla-annual-reports\\tesla-annual-reports\\tsla-10k_20201231-gen.pdf', 'source_doc': 'C:\\Users\\Priyanshi Garg\\Downloads\\tesla-annual-reports\\tesla-annual-reports\\tsla-10k_20201231-gen.pdf', 'title': '', 'total_pages': 449}, page_content='may\tarise\tduring\tour\tproduction\tramps,\tand\twe\tmust\taddress\tthem\tpromptly\twhile\tcontinuing\tto\timprove\tmanufacturing\tprocesses\tand\treducing\tcosts.\tIf\twe\nare\tnot\tsuccessful\tin\tachieving\tthese\tgoals,\twe\tcould\tface\tdelays\tin\testablishing\tand/or\tsustaining\tour\tModel\t3\tand\tModel\tY\tramps\tor\tbe\tunable\tto\tmeet\tour\nrelated\tcost\tand\tprofitability\ttargets.\nWe\tmay\talso\texperience\tsimil

In [30]:
for doc in results:

    print(doc.metadata)

    print(doc.page_content[:300])

    print("="*100)

{'chunk_id': 'tsla_2020_41', 'company': 'Tesla, Inc.', 'creationdate': '2021-02-08T12:40:35+00:00', 'creator': 'wkhtmltopdf 0.12.5', 'fiscal_year': 2020, 'form_type': '10-K', 'page': 13, 'page_label': '14', 'producer': 'Qt 5.11.3', 'source': 'C:\\Users\\Priyanshi Garg\\Downloads\\tesla-annual-reports\\tesla-annual-reports\\tsla-10k_20201231-gen.pdf', 'source_doc': 'C:\\Users\\Priyanshi Garg\\Downloads\\tesla-annual-reports\\tesla-annual-reports\\tsla-10k_20201231-gen.pdf', 'title': '', 'total_pages': 449}
may	arise	during	our	production	ramps,	and	we	must	address	them	promptly	while	continuing	to	improve	manufacturing	processes	and	reducing	costs.	If	we
are	not	successful	in	achieving	these	goals,	we	could	face	delays	in	establishing	and/or	sustaining	our	Model	3	and	Model	Y	ramps	or	be	unable	to	mee
{'chunk_id': 'tsla_2022_64', 'company': 'Tesla, Inc.', 'creationdate': '2023-01-31T11:10:39+00:00', 'creator': 'wkhtmltopdf 0.12.6', 'fiscal_year': 2022, 'form_type': '10-K', 'page': 19, '

### Baseline Answer Generation

In [31]:
context = "\n\n".join(
    [doc.page_content for doc in results]
)

In [32]:
prompt = f"""
Question:
{query}

Context:
{context}

Instructions:
- Answer only from context
- Cite chunk_id and year
"""

answer = client.chat.completions.create(
    model=model,
    temperature=0,
    messages=[
        {
            "role":"system",
            "content":"You are a financial analyst."
        },
        {
            "role":"user",
            "content":prompt
        }
    ]
)

print(answer.choices[0].message.content)

Based on the provided context, the major risks that could affect Tesla's production and delivery targets are:

1. **Manufacturing process improvements and cost-down efforts**: Delays in establishing and/or sustaining production ramps, or inability to meet cost and profitability targets (Chunk 14, no specific year mentioned).
2. **Employee retention and hiring**: Loss of key employees, including Elon Musk, or inability to attract and retain qualified personnel, which could disrupt operations or delay product development (Chunk 14, no specific year mentioned).
3. **Public credibility and confidence**: Negative perceptions, criticism, or speculation about Tesla's management team, products, or business prospects, which could harm the company's reputation and make it harder to raise funds (Chunk 18, no specific year mentioned).
4. **Supply chain and production bottlenecks**: Delays or complications in launching and/or ramping production of new vehicles, energy storage products, or Solar Roo

In [34]:
BATCH_SIZE = 100

chunk_batches = []

for i in range(0, len(tesla_chunks), BATCH_SIZE):

    batch = tesla_chunks[i:i + BATCH_SIZE]

    chunk_batches.append(batch)

print(f"Total chunks: {len(tesla_chunks)}")
print(f"Total batches: {len(chunk_batches)}")

Total chunks: 3337
Total batches: 34


In [36]:
chunk_batches[0][0].metadata

{'producer': 'Qt 5.11.3',
 'creator': 'wkhtmltopdf 0.12.5',
 'creationdate': '2020-09-30T05:19:35+00:00',
 'title': '',
 'source': 'C:\\Users\\Priyanshi Garg\\Downloads\\tesla-annual-reports\\tesla-annual-reports\\tsla-10k_20191231-gen_0.pdf',
 'total_pages': 297,
 'page': 0,
 'page_label': '1',
 'chunk_id': 'tsla_2019_0',
 'source_doc': 'C:\\Users\\Priyanshi Garg\\Downloads\\tesla-annual-reports\\tesla-annual-reports\\tsla-10k_20191231-gen_0.pdf',
 'company': 'Tesla, Inc.',
 'fiscal_year': 2019,
 'form_type': '10-K'}

In [35]:
hypothetical_questions_system_message = """
You will receive multiple Tesla 10-K document chunks.

Generate exactly 5 hypothetical questions that best represent
the information contained across all provided chunks.

Requirements:
- Questions should be useful for retrieval.
- Questions must be answerable from the provided chunks.
- Include a mix of:
    * factual
    * analytical
    * risk-oriented
    * operational
    * financial questions
- Do NOT introduce facts not present in the chunks.
- Generate exactly 5 questions.
- One question per line(e.g. "What is Tesla's revenue growth?").

Return ONLY valid JSON.

{
  "questions": [
    "...",
    "...",
    "...",
    "...",
    "..."
  ]
}
"""

In [37]:
import json

In [56]:
batch_questions = []

for batch_num, batch in enumerate(chunk_batches, start=1):

    try:

        print(
            f"Processing Batch {batch_num}/{len(chunk_batches)}"
        )

        batch_text = "\n\n".join(
            [
                f"Chunk {idx}: {chunk.page_content}"
                for idx, chunk in enumerate(batch, start=1)
            ]
        )

        response = client.chat.completions.create(
            model=model,
            temperature=0,
            messages=[
                {
                    "role": "system",
                    "content": hypothetical_questions_system_message
                },
                {
                    "role": "user",
                    "content": batch_text
                }
            ]
        )

        result = json.loads(
            response.choices[0].message.content
        )

        batch_questions.append({
            "batch_id": batch_num,
            "chunk_ids": [
                chunk.metadata["chunk_id"]
                for chunk in batch
            ],
            "questions": result["questions"]
        })

        print(
            f"✓ Batch {batch_num} Complete"
        )

    except Exception as e:

        print(
            f"✗ Batch {batch_num} Failed: {e}"
        )

Processing Batch 1/34
✓ Batch 1 Complete
Processing Batch 2/34
✓ Batch 2 Complete
Processing Batch 3/34
✓ Batch 3 Complete
Processing Batch 4/34
✓ Batch 4 Complete
Processing Batch 5/34
✓ Batch 5 Complete
Processing Batch 6/34
✓ Batch 6 Complete
Processing Batch 7/34
✓ Batch 7 Complete
Processing Batch 8/34
✓ Batch 8 Complete
Processing Batch 9/34
✓ Batch 9 Complete
Processing Batch 10/34
✓ Batch 10 Complete
Processing Batch 11/34
✓ Batch 11 Complete
Processing Batch 12/34
✓ Batch 12 Complete
Processing Batch 13/34
✓ Batch 13 Complete
Processing Batch 14/34
✓ Batch 14 Complete
Processing Batch 15/34
✓ Batch 15 Complete
Processing Batch 16/34
✓ Batch 16 Complete
Processing Batch 17/34
✓ Batch 17 Complete
Processing Batch 18/34
✓ Batch 18 Complete
Processing Batch 19/34
✓ Batch 19 Complete
Processing Batch 20/34
✓ Batch 20 Complete
Processing Batch 21/34
✓ Batch 21 Complete
Processing Batch 22/34
✓ Batch 22 Complete
Processing Batch 23/34
✓ Batch 23 Complete
Processing Batch 24/34
✓ Batc

In [57]:
with open(
    "batch_hypothetical_questions.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        batch_questions,
        f,
        indent=2,
        ensure_ascii=False
    )

In [16]:
import time

In [58]:
chunk_lookup = {
    chunk.metadata["chunk_id"]: chunk
    for chunk in tesla_chunks
}

In [59]:
print(chunk_lookup["tsla_2019_0"])

page_content='UNITED	STATES
SECURITIES	AND	EXCHANGE	COMMISSION
Washington,	D.C.	20549
	
FORM	
10-K
	
(Mark	One)
☒
ANNUAL	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	fiscal	year	ended	
December	31,	2019
	
OR
☐
TRANSITION	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	transition	period	from	
																				
	to	
																					
Commission	File	Number:	
001-34756
	
Tesla,	Inc.
(Exact	name	of	registrant	as	specified	in	its	charter)
	
	
Delaware
	
91-2197729
(State	or	other	jurisdiction	of
incorporation	or	organization)
	
(I.R.S.	Employer
Identification	No.)
	
	
	
3500	Deer	Creek	Road
Palo	Alto
,	
California
	
94304
(Address	of	principal	executive	offices)
	
(Zip	Code)
(
650
)	
681-5000
(Registrant’s	telephone	number,	including	area	code)
Securities	registered	pursuant	to	Section	12(b)	of	the	Act:
	
Title	of	each	class
Trading	Symbol(s)
Name	of	each	exchange	on	which	registered
Common	stock
TSLA
The	

In [72]:
import json
from langchain_core.documents import Document

hq_documents = []

for batch in batch_questions:

    for question in batch["questions"]:

        hq_documents.append(
            Document(
                page_content=question,
                metadata={
                    "batch_id": batch["batch_id"],
                    "chunk_ids": json.dumps(batch["chunk_ids"])
                }
            )
        )

In [64]:
print(hq_documents[0])

page_content='What is Tesla's business model and how does it differentiate itself from other companies?' metadata={'batch_id': 1, 'chunk_ids': '["tsla_2019_0", "tsla_2019_1", "tsla_2019_2", "tsla_2019_3", "tsla_2019_4", "tsla_2019_5", "tsla_2019_6", "tsla_2019_7", "tsla_2019_8", "tsla_2019_9", "tsla_2019_10", "tsla_2019_11", "tsla_2019_12", "tsla_2019_13", "tsla_2019_14", "tsla_2019_15", "tsla_2019_16", "tsla_2019_17", "tsla_2019_18", "tsla_2019_19", "tsla_2019_20", "tsla_2019_21", "tsla_2019_22", "tsla_2019_23", "tsla_2019_24", "tsla_2019_25", "tsla_2019_26", "tsla_2019_27", "tsla_2019_28", "tsla_2019_29", "tsla_2019_30", "tsla_2019_31", "tsla_2019_32", "tsla_2019_33", "tsla_2019_34", "tsla_2019_35", "tsla_2019_36", "tsla_2019_37", "tsla_2019_38", "tsla_2019_39", "tsla_2019_40", "tsla_2019_41", "tsla_2019_42", "tsla_2019_43", "tsla_2019_44", "tsla_2019_45", "tsla_2019_46", "tsla_2019_47", "tsla_2019_48", "tsla_2019_49", "tsla_2019_50", "tsla_2019_51", "tsla_2019_52", "tsla_2019_53", "

In [65]:
hq_vectorstore = Chroma.from_documents(
    documents=hq_documents,
    embedding=embedding_model,
    collection_name="tesla_hq",
    persist_directory="./chroma_hq_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [66]:
query = """
What should a board member ask about risks that could prevent Tesla from meeting production, delivery, or scaling expectations?
"""

In [67]:
hq_results = hq_vectorstore.similarity_search_with_score(
    query,
    k=10
)

In [68]:
hq_results[0]

(Document(id='2ee1be94-881b-4337-947d-7d5af1610e22', metadata={'batch_id': 30, 'chunk_ids': '["tsla_2022_673", "tsla_2022_674", "tsla_2022_675", "tsla_2022_676", "tsla_2022_677", "tsla_2022_678", "tsla_2022_679", "tsla_2022_680", "tsla_2022_681", "tsla_2022_682", "tsla_2022_683", "tsla_2022_684", "tsla_2022_685", "tsla_2022_686", "tsla_2022_687", "tsla_2022_688", "tsla_2022_689", "tsla_2022_690", "tsla_2022_691", "tsla_2022_692", "tsla_2022_693", "tsla_2022_694", "tsla_2022_695", "tsla_2022_696", "tsla_2022_697", "tsla_2022_698", "tsla_2022_699", "tsla_2022_700", "tsla_2022_701", "tsla_2022_702", "tsla_2022_703", "tsla_2022_704", "tsla_2022_705", "tsla_2022_706", "tsla_2022_707", "tsla_2022_708", "tsla_2022_709", "tsla_2022_710", "tsla_2022_711", "tsla_2022_712", "tsla_2022_713", "tsla_2022_714", "tsla_2022_715", "tsla_2022_716", "tsla_2022_717", "tsla_2022_718", "tsla_2022_719", "tsla_2022_720", "tsla_2022_721", "tsla_2022_722", "tsla_2022_723", "tsla_2022_724", "tsla_2022_725", "tsla

In [69]:
for doc, score in hq_results:

    print(doc.page_content)

    print(doc.metadata)

    print(score)

    print("=" * 80)

What are the risks associated with Tesla's business, as outlined in the risk factors section of the 10-K document?
{'batch_id': 30, 'chunk_ids': '["tsla_2022_673", "tsla_2022_674", "tsla_2022_675", "tsla_2022_676", "tsla_2022_677", "tsla_2022_678", "tsla_2022_679", "tsla_2022_680", "tsla_2022_681", "tsla_2022_682", "tsla_2022_683", "tsla_2022_684", "tsla_2022_685", "tsla_2022_686", "tsla_2022_687", "tsla_2022_688", "tsla_2022_689", "tsla_2022_690", "tsla_2022_691", "tsla_2022_692", "tsla_2022_693", "tsla_2022_694", "tsla_2022_695", "tsla_2022_696", "tsla_2022_697", "tsla_2022_698", "tsla_2022_699", "tsla_2022_700", "tsla_2022_701", "tsla_2022_702", "tsla_2022_703", "tsla_2022_704", "tsla_2022_705", "tsla_2022_706", "tsla_2022_707", "tsla_2022_708", "tsla_2022_709", "tsla_2022_710", "tsla_2022_711", "tsla_2022_712", "tsla_2022_713", "tsla_2022_714", "tsla_2022_715", "tsla_2022_716", "tsla_2022_717", "tsla_2022_718", "tsla_2022_719", "tsla_2022_720", "tsla_2022_721", "tsla_2022_722", "ts

In [70]:
parent_chunks = []

seen = set()

In [74]:
for doc, score in hq_results[:1]:
    print(repr(doc.metadata["chunk_ids"]))

'["tsla_2022_673", "tsla_2022_674", "tsla_2022_675", "tsla_2022_676", "tsla_2022_677", "tsla_2022_678", "tsla_2022_679", "tsla_2022_680", "tsla_2022_681", "tsla_2022_682", "tsla_2022_683", "tsla_2022_684", "tsla_2022_685", "tsla_2022_686", "tsla_2022_687", "tsla_2022_688", "tsla_2022_689", "tsla_2022_690", "tsla_2022_691", "tsla_2022_692", "tsla_2022_693", "tsla_2022_694", "tsla_2022_695", "tsla_2022_696", "tsla_2022_697", "tsla_2022_698", "tsla_2022_699", "tsla_2022_700", "tsla_2022_701", "tsla_2022_702", "tsla_2022_703", "tsla_2022_704", "tsla_2022_705", "tsla_2022_706", "tsla_2022_707", "tsla_2022_708", "tsla_2022_709", "tsla_2022_710", "tsla_2022_711", "tsla_2022_712", "tsla_2022_713", "tsla_2022_714", "tsla_2022_715", "tsla_2022_716", "tsla_2022_717", "tsla_2022_718", "tsla_2022_719", "tsla_2022_720", "tsla_2022_721", "tsla_2022_722", "tsla_2022_723", "tsla_2022_724", "tsla_2022_725", "tsla_2022_726", "tsla_2022_727", "tsla_2022_728", "tsla_2022_729", "tsla_2022_730", "tsla_2022_7

In [77]:
import json

parent_chunks = []
seen = set()

for doc, score in hq_results:
    chunk_ids_raw = doc.metadata.get("chunk_ids")

    if chunk_ids_raw:
        try:
            chunk_ids = json.loads(chunk_ids_raw)
        except (TypeError, json.JSONDecodeError):
            chunk_ids = [chunk_ids_raw] if isinstance(chunk_ids_raw, str) else []
    else:
        chunk_ids = [doc.metadata.get("chunk_id")] if doc.metadata.get("chunk_id") else []

    for chunk_id in chunk_ids:
        if chunk_id and chunk_id not in seen:
            seen.add(chunk_id)
            if chunk_id in chunk_lookup:
                parent_chunks.append(chunk_lookup[chunk_id])


In [78]:
print(len(parent_chunks))

print(parent_chunks[0].metadata)

500
{'producer': 'Qt 5.15.2', 'creator': 'wkhtmltopdf 0.12.6', 'creationdate': '2023-01-31T11:10:39+00:00', 'title': '', 'source': 'C:\\Users\\Priyanshi Garg\\Downloads\\tesla-annual-reports\\tesla-annual-reports\\tsla-20221231-gen.pdf', 'total_pages': 251, 'page': 204, 'page_label': '205', 'chunk_id': 'tsla_2022_673', 'source_doc': 'C:\\Users\\Priyanshi Garg\\Downloads\\tesla-annual-reports\\tesla-annual-reports\\tsla-20221231-gen.pdf', 'company': 'Tesla, Inc.', 'fiscal_year': 2022, 'form_type': '10-K'}


In [79]:
context = "\n\n".join(
    [
        chunk.page_content
        for chunk in parent_chunks
    ]
)

In [80]:
prompt = f"""
Question:
{query}

Context:
{context}

Instructions:
- Answer only from context
- Include citations
- Mention chunk_id and fiscal_year
"""

In [81]:
response = client.chat.completions.create(
    model=model,
    temperature=0,
    messages=[
        {
            "role": "system",
            "content": "You are a financial analyst."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
)

final_answer = response.choices[0].message.content

print(final_answer)

The board of directors of Tesla, Inc. has adopted a policy requiring the company to obtain the approval of the board of directors or a committee thereof before entering into any transaction with a related party. This policy is intended to ensure that the company's interests are protected and that any transactions with related parties are fair and reasonable.

The policy defines a related party as any of the following: (i) a director or officer of the company, (ii) a significant shareholder of the company, or (iii) a spouse, child, sibling, parent, or affiliate of any of the foregoing. A significant shareholder is defined as a shareholder who owns 5% or more of the company's outstanding common stock.

The policy requires that the board of directors or a committee thereof must approve any transaction with a related party before it is entered into. This approval must be obtained in writing and must be recorded in the minutes of the meeting at which the approval is given.

The policy also 

In [82]:
output = {
    "question_id": "HQ1",
    "user_query": query,
    "retrieved_hypothetical_questions": [
        {
            "question": doc.page_content,
            "batch_id": doc.metadata["batch_id"]
        }
        for doc, score in hq_results
    ],
    "final_answer": final_answer
}

In [83]:
with open(
    "HQ1_output.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        indent=2,
        ensure_ascii=False
    )